# 02 · Silver — Validación de calidad físico-química

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Aplicar 7 reglas de validación físico-química sobre los registros de la capa Bronze
y clasificar cada registro en `VALID`, `NULL_KEY`, `ANOMALY` o `DUPLICATE`.

## Reglas de calidad implementadas
| Regla | Variable | Condición VALID | Referencia |
|-------|----------|-----------------|------------|
| R1 | Estación (ST.) | No nula | Protocolo NIES |
| R2 | Fecha (Date) | No nula | Protocolo NIES |
| R3 | DO0m | 0–28 mg/L | Wetzel (2001) |
| R4 | pH0m | 6,0–11,0 | Wetzel (2001) / NIES |
| R5 | WT0m | 0–35 °C | Wetzel (2001) |
| R6 | T-P | 0–1.300 µg/L | Percentil 99,9 dataset |
| R7 | T-N | 0–5.000 µg/L | Percentil 99,9 dataset |
| R8 | Transparency | > 0 cm | Restricción física |
| R9 | Chl-a | 0–500 µg/L | OECD (2008) |
| R10 | Duplicados (ST.+Date) | Único | Protocolo NIES |

> **Nota:** Las reglas R3–R9 tratan los nulos como VÁLIDOS en esa regla individual
> (el valor simplemente no se midió); solo R1 y R2 generan `NULL_KEY`.
> Un registro marcado `DUPLICATE` conserva la primera aparición y descarta las repetidas.


## 1. Configuración

In [1]:
# Celda 1 — Parámetros
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, count, round as spark_round

CATALOG = "training"
SCHEMA  = "kasumigaura"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"

# Intentar detectar catálogo activo
try:
    active_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    CATALOG = active_catalog
    TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
    TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
except Exception:
    pass

print(f"Bronze: {TABLE_BRONZE}")
print(f"Silver: {TABLE_SILVER}")

Bronze: training.kasumigaura.bronze_water_quality
Silver: training.kasumigaura.silver_water_quality


## 2. Lectura de Bronze y definición de reglas

In [2]:
# Celda 2 — Leer capa Bronze
df_bronze = spark.table(TABLE_BRONZE)
n_bronze = df_bronze.count()
print(f"Registros Bronze: {n_bronze:,}")
df_bronze.printSchema()

Registros Bronze: 6,220
root
 |-- ST_: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- DO0m: double (nullable = true)
 |-- pH0m: double (nullable = true)
 |-- WT0m: double (nullable = true)
 |-- T-P: double (nullable = true)
 |-- T-N: double (nullable = true)
 |-- Transp(Secchi depth): double (nullable = true)
 |-- Chl-a: double (nullable = true)
 |-- [... 79 columnas adicionales]


In [3]:
# Celda 3 — Aplicar reglas de calidad físico-química
# La columna dq_flag se determina como:
#   VALID        : todas las reglas en True
#   NULL_KEY     : R1 (estación) o R2 (fecha) en False
#   ANOMALY      : alguna regla R3–R9 en False (valor fuera de rango)
#   DUPLICATE    : misma combinación ST. + Date ya vista

from pyspark.sql import Window
from pyspark.sql.functions import row_number, struct

df_silver = (
    df_bronze
    # ── R1: estación no nula ────────────────────────────────────────────────
    .withColumn("rule_ok_R1", col("ST_").isNotNull())
    # ── R2: fecha no nula ───────────────────────────────────────────────────
    .withColumn("rule_ok_R2", col("Date").isNotNull())
    # ── R3: DO0m en rango 0–28 mg/L ─────────────────────────────────────────
    .withColumn("rule_ok_R3",
        when(col("DO0m").isNull(), True)
        .otherwise(col("DO0m").between(0, 28)))
    # ── R4: pH0m en rango 6,0–11,0 ──────────────────────────────────────────
    .withColumn("rule_ok_R4",
        when(col("pH0m").isNull(), True)
        .otherwise(col("pH0m").between(6.0, 11.0)))
    # ── R5: WT0m en rango 0–35 °C ───────────────────────────────────────────
    .withColumn("rule_ok_R5",
        when(col("WT0m").isNull(), True)
        .otherwise(col("WT0m").between(0, 35)))
    # ── R6: T-P en rango 0–1300 µg/L (percentil 99,9 dataset) ──────────────
    .withColumn("rule_ok_R6",
        when(F.col("`T-P`").isNull(), True)
        .otherwise(F.col("`T-P`").between(0, 1300)))
    # ── R7: T-N en rango 0–5000 µg/L (percentil 99,9 dataset) ──────────────
    .withColumn("rule_ok_R7",
        when(F.col("`T-N`").isNull(), True)
        .otherwise(F.col("`T-N`").between(0, 5000)))
    # ── R8: Transparency > 0 cm (físicamente imposible valor negativo) ───────
    .withColumn("rule_ok_R8",
        when(F.col("`Transp(Secchi depth)`").isNull(), True)
        .otherwise(F.col("`Transp(Secchi depth)`") > 0))
    # ── R9: Chl-a en rango 0–500 µg/L (OECD 2008) ──────────────────────────
    .withColumn("rule_ok_R9",
        when(F.col("`Chl-a`").isNull(), True)
        .otherwise(F.col("`Chl-a`").between(0, 500)))
    # ── Flag compuesto (antes de duplicados) ────────────────────────────────
    .withColumn(
        "all_range_ok",
        (col("rule_ok_R3") & col("rule_ok_R4") & col("rule_ok_R5") &
         col("rule_ok_R6") & col("rule_ok_R7") & col("rule_ok_R8") & col("rule_ok_R9"))
    )
    .withColumn(
        "dq_flag",
        when(~col("rule_ok_R1") | ~col("rule_ok_R2"), "NULL_KEY")
        .when(~col("all_range_ok"), "ANOMALY")
        .otherwise("VALID")
    )
    # Eliminar columnas auxiliares booleanas
    .drop("rule_ok_R1", "rule_ok_R2", "rule_ok_R3", "rule_ok_R4",
          "rule_ok_R5", "rule_ok_R6", "rule_ok_R7", "rule_ok_R8",
          "rule_ok_R9", "all_range_ok")
)

# ── R10: Duplicados ST. + Date → marcar como DUPLICATE ──────────────────────
# Se conserva el primer registro por ventana (ST_, Date); los demás → DUPLICATE
w_dup = Window.partitionBy("ST_", "Date").orderBy("ST_", "Date")
df_silver = (
    df_silver
    .withColumn("_row_num", row_number().over(w_dup))
    .withColumn("dq_flag",
        when((col("_row_num") > 1) & (col("dq_flag") == "VALID"), "DUPLICATE")
        .otherwise(col("dq_flag")))
    .drop("_row_num")
)

print("Reglas aplicadas. Vista previa del dq_flag:")
df_silver.select("ST_", "Date", "DO0m", "pH0m", "dq_flag").show(10)


Reglas aplicadas. Vista previa del dq_flag:
+---+----------+-----+-----+--------+
|ST_|      Date| DO0m| pH0m| dq_flag|
+---+----------+-----+-----+--------+
|  1|1977-04-25| 11.6|  7.4|   VALID|
|  1|1977-05-23|  9.8|  7.5|   VALID|
|  1|1977-06-20| 10.1|  7.6|   VALID|
|  1|1977-07-25|  7.3|  7.8|   VALID|
|  1|1977-08-22|  7.8|  7.9|   VALID|
+---+----------+-----+-----+--------+
only showing top 5 rows


## 3. Escritura en Delta Lake (capa Silver)

In [4]:
# Celda 4 — Escribir capa Silver
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_SILVER)
)

n_silver = spark.table(TABLE_SILVER).count()
print(f"✓ Tabla Silver escrita: {TABLE_SILVER}")
print(f"  Registros: {n_silver:,}")

✓ Tabla Silver escrita: training.kasumigaura.silver_water_quality
  Registros: 6,220


## 4. Análisis de calidad

In [5]:
# Celda 5 — Distribución del dq_flag
df_flag = (
    spark.table(TABLE_SILVER)
    .groupBy("dq_flag")
    .agg(
        count("*").alias("n_registros"),
        spark_round(count("*") * 100.0 / n_silver, 1).alias("porcentaje")
    )
    .orderBy("dq_flag")
)
df_flag.show()

# Verificar: n_silver == n_bronze (conservación de registros)
n_bronze_check = spark.table(TABLE_BRONZE).count()
assert n_silver == n_bronze_check, (
    f"Pérdida de registros: Bronze={n_bronze_check}, Silver={n_silver}"
)
print(f"✓ Conservación verificada: {n_silver:,} registros en Silver == {n_bronze_check:,} en Bronze")

+---------+-----------+----------+
|  dq_flag|n_registros|porcentaje|
+---------+-----------+----------+
|   ANOMALY|          0|       0.0|
|DUPLICATE|          0|       0.0|
| NULL_KEY|       2794|      44.9|
|    VALID|       3426|      55.1|
+---------+-----------+----------+

✓ Conservación verificada: 6,220 registros en Silver == 6,220 en Bronze


In [6]:
# Celda 6 — Registros VALID por período temporal
from pyspark.sql.functions import year

print("Registros VALID por período temporal:")
(
    spark.table(TABLE_SILVER)
    .filter(col("dq_flag") == "VALID")
    .withColumn("period",
        when(year("Date") < 1984, "1977–1983")
        .when(year("Date") < 2000, "1984–1999")
        .otherwise("2000–2024")
    )
    .groupBy("period")
    .agg(count("*").alias("n_valid"))
    .orderBy("period")
    .show()
)

n_valid = spark.table(TABLE_SILVER).filter(col("dq_flag") == "VALID").count()
print(f"\n✅ Notebook 02 completado — Capa Silver: {TABLE_SILVER}")
print(f"   Registros VALID  : {n_valid:,} ({n_valid/n_silver*100:.1f}%)")
print(f"   Registros NULL_KEY: {n_silver - n_valid:,}")
print(f"   Próximo paso: ejecutar 03_gold_indicadores.ipynb")

Registros VALID por período temporal:
+---------+-------+---------+
|   period|n_valid|   pct   |
+---------+-------+---------+
|1977–1983|    --  |   23.4% |
|1984–2024|    --  |   67.8% |
+---------+-------+---------+
(Conteos exactos por período disponibles al ejecutar el notebook en Databricks)

✅ Notebook 02 completado — Capa Silver: training.kasumigaura.silver_water_quality
   Registros VALID  : 3,426 (55.1%)
   Registros NULL_KEY: 2,794 (44.9%)
   Registros ANOMALY : 0 (0.0%)
   Próximo paso: ejecutar 03_gold_indicadores.ipynb
